In [1]:
from datasets import load_dataset
import json
path="VLFFD-main/output.json"
data = load_dataset("json", data_files=path, split="train")

data = data.train_test_split(
    train_size=0.9, test_size=0.1, seed=42, shuffle=True
)
data["validation"] = data.pop("test")
data

DatasetDict({
    train: Dataset({
        features: ['question', 'imagepath', 'answer', 'reason'],
        num_rows: 3870
    })
    validation: Dataset({
        features: ['question', 'imagepath', 'answer', 'reason'],
        num_rows: 430
    })
})

In [2]:
from typing import Any
from PIL import Image
def format_data(example:dict[str,Any])-> dict[str, Any]:
    example["messages"]=[
        {
            "role": "user", 
            "content": [
                {
                    "type":"image",
                },
                {
                    "type":"text",
                    "text":example["question"],
                },
            ],
        },
    ]
    example["image"] = Image.open(example["imagepath"])
    return example
data = data.map(format_data, remove_columns=["question","imagepath"])
data["train"][0]

Map:   0%|          | 0/3870 [00:00<?, ? examples/s]

Map:   0%|          | 0/430 [00:00<?, ? examples/s]

{'answer': 'fake',
 'reason': 'This is a fake face. The skin texture appears unnaturally smooth and uniform, lacking the subtle variations found in real skin. The eyes region shows that the eyes appear blurry compared to natural faces, and the eyes exhibits unusual texture patterns, and the eyes appears structurally distorted. The nose region shows that the nose exhibits unusual texture patterns. The mouth region shows that the mouth appears structurally distorted. The lighting is somewhat diffuse and lacks strong shadows. The eyebrows also appear unnatural, with an overly defined and artificial shape.',
 'messages': [{'content': [{'text': None, 'type': 'image'},
    {'text': 'Has this face been digitally altered?', 'type': 'text'}],
   'role': 'user'}],
 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=256x256>}

In [ ]:
from evaluate import load

model_id="google/gemma-3-4b-it"
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
rouge_metric = load("rouge")  # 文本生成常用ROUGE指标
bleu_metric = load("bleu")    # 或BLEU指标

REFERENCES = data["train"]["answer"]

def compute_text_metrics(predictions):
    
    # 解码token ID为文本
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(REFERENCES, skip_special_tokens=True)

    # 计算ROUGE分数
    rouge_score = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    
    # 计算BLEU分数
    bleu_score = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[ref] for ref in decoded_labels]  # BLEU需要引用文本列表
    )
    
    return {
        "rouge1": rouge_score["rouge1"],
        "rougeL": rouge_score["rougeL"],
        "bleu": bleu_score["bleu"]
    }


In [11]:
from transformers.pipelines import pipeline
import torch
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(model_id)
pt_pipe = pipeline(
    "image-text-to-text",
    model=model_id,
    torch_dtype=torch.bfloat16,
)

# Set `do_sample = False` for deterministic responses
pt_pipe.model.generation_config.do_sample = False
pt_pipe.model.generation_config.pad_token_id = processor.tokenizer.eos_token_id

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use mps:0


In [ ]:
pt_outputs = pt_pipe(
    text=data["train"]["messages"],
    images=data["train"]["image"],
    max_new_tokens=400,
    batch_size=32,
    return_full_text=False,
)

pt_predictions = [out for out in pt_outputs]

In [ ]:
pt_metrics = compute_text_metrics(pt_predictions)
print(f"Baseline metrics: {pt_metrics}")

Compute metrics on the fine-tuned model
Load the base model with the fine-tuned LoRA adapter using the pipeline API.

In [ ]:
args = SFTConfig(
    output_dir="./output",            # Directory and Hub repository id to save the model to
    num_train_epochs=num_train_epochs,                       # Number of training epochs
    per_device_train_batch_size=2,                           # Batch size per device during training
    per_device_eval_batch_size=2,                            # Batch size per device during evaluation
    gradient_accumulation_steps=2,                           # Number of steps before performing a backward/update pass
    gradient_checkpointing=True,                             # Enable gradient checkpointing to reduce memory usage
    optim="adamw_torch_fused",                               # Use fused AdamW optimizer for better performance
    logging_steps=50,                                        # Number of steps between logs
    save_strategy="epoch",                                   # Save checkpoint every epoch
    eval_strategy="steps",                                   # Evaluate every `eval_steps`
    eval_steps=50,                                           # Number of steps between evaluations
    learning_rate=learning_rate,                             # Learning rate based on QLoRA paper
    # bf16=True,                                               # Use bfloat16 precision
    max_grad_norm=0.3,                                       # Max gradient norm based on QLoRA paper
    warmup_ratio=0.03,                                       # Warmup ratio based on QLoRA paper
    lr_scheduler_type="linear",                              # Use linear learning rate scheduler
    push_to_hub=False,                                        # Push model to Hub
    report_to="tensorboard",                                 # Report metrics to tensorboard
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Set gradient checkpointing to non-reentrant to avoid issues
    dataset_kwargs={"skip_prepare_dataset": True},           # Skip default dataset preparation to preprocess manually
    remove_unused_columns = False,                           # Columns are unused for training but needed for data collator
    label_names=["labels"],                                  # Input keys that correspond to the labels
)

In [ ]:
ft_pipe = pipeline(
    "image-text-to-text",
    model=args.output_dir,
    processor=processor,
    torch_dtype=torch.bfloat16,
)

# Set `do_sample = False` for deterministic responses
ft_pipe.model.generation_config.do_sample = False
ft_pipe.model.generation_config.pad_token_id = processor.tokenizer.eos_token_id
# Use left padding during inference
processor.tokenizer.padding_side = "left"

In [ ]:
ft_outputs = ft_pipe(
    text=data["train"]["messages"],
    images=data["train"]["image"],
    max_new_tokens=400,
    batch_size=32,
    return_full_text=False,
)

ft_predictions = [out for out in ft_outputs]

In [ ]:
ft_metrics = compute_text_metrics(ft_predictions)
print(f"Fine-tuned metrics: {ft_metrics}")